In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, re, gc, argparse
from dataclasses import dataclass
from typing import List, Tuple, Optional, Sequence

import numpy as np
from tifffile import TiffFile, imwrite
from tqdm import tqdm

import torch
from torch import Tensor
from sklearn.neighbors import KernelDensity
import torch.nn.functional as F
from time import time

import os, math, hashlib
from concurrent.futures import ThreadPoolExecutor

# -------------------- basic utils --------------------
def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def find_tiffs_sorted_by_last_number(root: str) -> List[str]:
    paths = []
    for base, _, files in os.walk(root):
        for fn in files:
            if fn.lower().endswith((".tif",".tiff")):
                paths.append(os.path.join(base, fn))
    if not paths:
        raise FileNotFoundError(f"No TIFFs under {root}")
    def last_int(s: str) -> int:
        nums = re.findall(r"\d+", os.path.basename(s))
        return int(nums[-1]) if nums else -1
    paths.sort(key=last_int)
    return paths

def read_tiff_volume(path: str) -> np.ndarray:
    with TiffFile(path) as tf:
        arr = tf.asarray()
    return np.asarray(arr)

def matlab_range_to_py_slice(start_inclusive: int, end_inclusive: int) -> slice:
    return slice(start_inclusive - 1, end_inclusive)

def clamp_inclusive_range(a: int, b: int, lo: int, hi: int) -> Tuple[int,int]:
    a2 = max(lo, min(a, hi)); b2 = max(lo, min(b, hi))
    if a2 > b2: a2, b2 = b2, a2
    return a2, b2

def load_valid_mask(valid_mask_tif: str, Z: int, Y: int, X: int) -> np.ndarray:
    vm = read_tiff_volume(valid_mask_tif)

    # Accept either 2D (Y,X) or 3D (Z,Y,X)
    if vm.ndim == 2:
        if vm.shape != (Y, X):
            raise ValueError(f"Valid mask shape {vm.shape} != {(Y, X)}")
        vm2 = (vm > 0)
        return np.broadcast_to(vm2, (Z, Y, X))

    if vm.ndim == 3:
        if vm.shape[1:] != (Y, X):
            raise ValueError(f"Valid mask Y/X {vm.shape[1:]} != {(Y, X)}")
        if vm.shape[0] < Z:
            raise ValueError(f"Valid mask has only Z={vm.shape[0]} slices, need Z={Z}")
        return (vm[:Z] > 0)

    raise ValueError(f"Valid mask must be 2D or 3D TIFF, got ndim={vm.ndim}")

# -------------------- shape scan & alignment --------------------
@dataclass
class ShapeInfo:
    min_z: int
    y: int
    x: int
    per_file_z: List[Tuple[str,int]]

def scan_feature_shapes(feature_paths: List[str]) -> ShapeInfo:
    y_ref = x_ref = None
    per_file = []
    min_z = None
    for fp in feature_paths:
        with TiffFile(fp) as tf:
            z_i = len(tf.pages)
            y_i, x_i = tf.pages[0].asarray().shape
        if y_ref is None:
            y_ref, x_ref = int(y_i), int(x_i)
        else:
            if (y_i, x_i) != (y_ref, x_ref):
                raise ValueError(f"Y/X mismatch: {fp} has {(y_i,x_i)} vs {(y_ref,x_ref)}")
        per_file.append((fp, int(z_i)))
        min_z = int(z_i) if min_z is None else min(min_z, int(z_i))
    return ShapeInfo(min_z=min_z, y=y_ref, x=x_ref, per_file_z=per_file)


# -------------------- Stage 1: sampling --------------------

def _precompute_mask_indices(bg_mask: np.ndarray, fg_mask: np.ndarray):
    """
    Precompute linear indices per Z for BG and FG.
    This lets us sample k indices directly from tile.ravel() without building tile[mask].
    """
    Zc, H, W = bg_mask.shape
    bg_idx_per_z, fg_idx_per_z = [], []
    for z in range(Zc):
        bg_idx_per_z.append(np.flatnonzero(bg_mask[z].ravel()))
        fg_idx_per_z.append(np.flatnonzero(fg_mask[z].ravel()))
    return bg_idx_per_z, fg_idx_per_z

def _seed_for_feature(global_seed: int, feature_path: str) -> int:
    """
    Stable per-feature seed: combine global seed with a hash of the basename.
    This makes results reproducible across runs, independent of thread scheduling.
    """
    name = os.path.basename(feature_path).encode("utf-8")
    h = int.from_bytes(hashlib.blake2b(name, digest_size=8).digest(), "little")
    # keep in int64 range that numpy Generator accepts
    return int((np.uint64(global_seed) ^ np.uint64(h)) % np.uint64(2**63 - 1))

def collect_samples_all_features(
    feature_paths: List[str],
    bg_mask: np.ndarray,  # [Z,Y,X] boolean
    fg_mask: np.ndarray,  # [Z,Y,X] boolean
    max_samples_per_class: int,
    rng: np.random.Generator,
    *,
    io_workers: Optional[int] = None
):
    """
    Precision‑preserving fast sampler for Stage 1.

    - Uses ALL z-slices (no subsampling).
    - Targets the SAME per‑Z budget as your original code:
        per_z = ceil(max_samples_per_class / z_lim)   (per feature)
      and trims to the cap at the end (same behavior).
    - Parallelizes across features with a thread pool.
    """
    Zc = bg_mask.shape[0]
    bg_idx_per_z, fg_idx_per_z = _precompute_mask_indices(bg_mask, fg_mask)

    if io_workers is None:
        # Enough threads to utilize I/O and decompression, but avoid oversubscription
        io_workers = max(1, min(8, os.cpu_count() or 8))

    # Prebuild a deterministic per-feature seed (based on current global RNG + feature name)
    # This freezes randomness per feature regardless of thread interleaving.
    base_seed = int(rng.integers(0, 2**63 - 1, dtype=np.int64))
    feat_seeds = [
        _seed_for_feature(base_seed, fp)
        for fp in feature_paths
    ]

    def _worker(i: int, fp: str, seed: int):
        local_rng = np.random.default_rng(seed)
        bg_out, fg_out = [], []
        with TiffFile(fp) as tf:
            z_lim = min(Zc, len(tf.pages))
            # EXACTLY the same per-Z allocation rule as your original:
            per_z_bg = int(math.ceil(max_samples_per_class / max(1, z_lim)))
            per_z_fg = int(math.ceil(max_samples_per_class / max(1, z_lim)))

            for z in range(z_lim):
                tile = tf.pages[z].asarray()
                flat = tile.ravel()

                # BG
                idxs_bg = bg_idx_per_z[z]
                if idxs_bg.size:
                    kbg = min(per_z_bg, idxs_bg.size)
                    pick = local_rng.choice(idxs_bg, size=kbg, replace=False)
                    bg_out.append(flat[pick])

                # FG
                idxs_fg = fg_idx_per_z[z]
                if idxs_fg.size:
                    kfg = min(per_z_fg, idxs_fg.size)
                    pick = local_rng.choice(idxs_fg, size=kfg, replace=False)
                    fg_out.append(flat[pick])

        bg = np.concatenate(bg_out, dtype=np.float32) if bg_out else np.empty((0,), np.float32)
        fg = np.concatenate(fg_out, dtype=np.float32) if fg_out else np.empty((0,), np.float32)

        # If we overshot due to ceil() per Z, RANDOMLY TRIM to the cap
        # (same semantics as your code which trimmed after concatenation).
        if bg.size > max_samples_per_class:
            bg = bg[local_rng.permutation(bg.size)[:max_samples_per_class]]
        if fg.size > max_samples_per_class:
            fg = fg[local_rng.permutation(fg.size)[:max_samples_per_class]]

        return i, bg, fg

    bg_list = [None] * len(feature_paths)
    fg_list = [None] * len(feature_paths)
    with ThreadPoolExecutor(max_workers=io_workers) as ex:
        futures = [ex.submit(_worker, i, fp, feat_seeds[i]) for i, fp in enumerate(feature_paths)]
        for fut in futures:
            i, bg, fg = fut.result()
            bg_list[i] = bg
            fg_list[i] = fg
    return bg_list, fg_list

# -------------------- save/load caches (Stage 1 & 2) --------------------
def save_samples_npz(
    path: str,
    bg_list: Sequence[np.ndarray],
    fg_list: Sequence[np.ndarray],
    feature_paths: Sequence[str],
    *,
    compress: bool = True,
) -> None:
    """
    Save Stage 1 samples cache as NPZ.

    If compress=True  -> np.savez_compressed (smaller, slower)
       compress=False -> np.savez            (bigger, faster)
    """
    # Pack as object arrays (same structure you already use)
    bg_obj = np.array(bg_list, dtype=object)
    fg_obj = np.array(fg_list, dtype=object)
    meta_paths  = np.array(feature_paths, dtype=object)
    meta_names  = np.array([os.path.basename(p) for p in feature_paths], dtype=object)

    if compress:
        np.savez_compressed(
            path,
            bg=bg_obj,
            fg=fg_obj,
            feature_paths=meta_paths,
            feature_names=meta_names,
        )
    else:
        np.savez(
            path,
            bg=bg_obj,
            fg=fg_obj,
            feature_paths=meta_paths,
            feature_names=meta_names,
        )

def load_samples_npz(path: str):
    d = np.load(path, allow_pickle=True)
    bg_list = [np.asarray(x, dtype=np.float32) for x in d["bg"]]
    fg_list = [np.asarray(x, dtype=np.float32) for x in d["fg"]]
    fpaths = [str(x) for x in d["feature_paths"]]
    fnames = [str(x) for x in d["feature_names"]]
    return bg_list, fg_list, fpaths, fnames

def save_densities_npz(
    path: str,
    x1_all, f1_all, x2_all, f2_all,
    feature_paths: Sequence[str],
    *,
    compress: bool = True,
) -> None:
    arr = lambda L: np.array(L, dtype=object)
    meta_paths = np.array(feature_paths, dtype=object)
    meta_names = np.array([os.path.basename(p) for p in feature_paths], dtype=object)
    if compress:
        np.savez_compressed(
            path,
            x1=arr(x1_all), f1=arr(f1_all),
            x2=arr(x2_all), f2=arr(f2_all),
            feature_paths=meta_paths,
            feature_names=meta_names,
        )
    else:
        np.savez(
            path,
            x1=arr(x1_all), f1=arr(f1_all),
            x2=arr(x2_all), f2=arr(f2_all),
            feature_paths=meta_paths,
            feature_names=meta_names,
        )


def load_densities_npz(path: str):
    d = np.load(path, allow_pickle=True)
    x1_all = [np.asarray(x, dtype=np.float32) for x in d["x1"]]
    f1_all = [np.asarray(x, dtype=np.float32) for x in d["f1"]]
    x2_all = [np.asarray(x, dtype=np.float32) for x in d["x2"]]
    f2_all = [np.asarray(x, dtype=np.float32) for x in d["f2"]]
    fpaths = [str(x) for x in d["feature_paths"]]
    fnames = [str(x) for x in d["feature_names"]]
    return x1_all, f1_all, x2_all, f2_all, fpaths, fnames

def reorder_lists_to_current(current_paths: Sequence[str],
                             saved_paths: Sequence[str],
                             saved_names: Sequence[str],
                             lists: List[List[np.ndarray]]):
    """
    Try to reorder saved lists to match current feature_paths by basename.
    lists is a list of lists (e.g., [x1_all, f1_all, x2_all, f2_all]) each length F_saved.
    Returns reordered lists to length F_current.
    """
    curr_names = [os.path.basename(p) for p in current_paths]
    if len(curr_names) != len(saved_names):
        # try subset by intersecting names (conservative)
        raise RuntimeError("Feature count mismatch between cache and current run.")
    # mapping by unique basename
    if len(set(saved_names)) != len(saved_names) or len(set(curr_names)) != len(curr_names):
        # fall back to full path equality
        if list(current_paths) != list(saved_paths):
            raise RuntimeError("Cannot align cached features to current features (duplicates or different sets).")
        idx_order = list(range(len(current_paths)))
    else:
        pos = {name: i for i, name in enumerate(saved_names)}
        try:
            idx_order = [pos[name] for name in curr_names]
        except KeyError as e:
            raise RuntimeError(f"Cached features missing: {e}")
    out = []
    for arr_list in lists:
        out.append([arr_list[i] for i in idx_order])
    return out


# -------------------- Stage 2: fit densities (KDE) --------------------
def bw_auto(values: np.ndarray) -> float:
    v = np.asarray(values, dtype=np.float64)
    v = v[np.isfinite(v)]
    n = v.size
    if n < 2:
        return 0.2
    std = np.std(v)
    iqr = np.subtract(*np.percentile(v, [75,25]))
    sigma = min(std, iqr/1.349) if std > 0 else (iqr/1.349 if iqr > 0 else 1.0)
    return float(max(1e-6, 1.06 * sigma * (n ** (-1/5))))

def kde_pdf(values: np.ndarray, grid: np.ndarray, bandwidth: Optional[float]) -> np.ndarray:
    v = np.asarray(values, dtype=np.float64).reshape(-1,1)
    if v.size == 0:
        return np.zeros_like(grid, dtype=np.float64)
    bw = bw_auto(v) if bandwidth is None else float(bandwidth)
    kde = KernelDensity(kernel='gaussian', bandwidth=bw)
    kde.fit(v)
    logd = kde.score_samples(grid.reshape(-1,1))
    d = np.exp(logd)
    return np.maximum(d, 0.0)

def fit_densities_from_samples(bg_list: List[np.ndarray], fg_list: List[np.ndarray],
                               num_points: int, bandwidth: Optional[float]):
    assert len(bg_list) == len(fg_list)
    x1_all, f1_all, x2_all, f2_all = [], [], [], []
    print("[info] Stage 2: fitting per-feature densities (KDE) ...")
    for bg_vals, fg_vals in tqdm(list(zip(bg_list, fg_list)), total=len(bg_list), desc="KDE"):
        if bg_vals.size and fg_vals.size:
            vmin = float(min(bg_vals.min(), fg_vals.min()))
            vmax = float(max(bg_vals.max(), fg_vals.max()))
        elif bg_vals.size:
            vmin, vmax = float(bg_vals.min()), float(bg_vals.max())
        elif fg_vals.size:
            vmin, vmax = float(fg_vals.min()), float(fg_vals.max())
        else:
            vmin, vmax = 0.0, 1.0
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = 0.0, 1.0
        grid = np.linspace(vmin, vmax, num_points, dtype=np.float64)
        f_bg = kde_pdf(bg_vals, grid, bandwidth)
        f_fg = kde_pdf(fg_vals, grid, bandwidth)
        x1_all.append(grid.astype(np.float32)); f1_all.append(f_bg.astype(np.float32))
        x2_all.append(grid.astype(np.float32)); f2_all.append(f_fg.astype(np.float32))
    return x1_all, f1_all, x2_all, f2_all

def _gaussian_kernel_1d(sigma_bins: float, device: torch.device) -> torch.Tensor:
    # 3σ on each side covers >99% mass
    K = int(max(3, 2*round(3*sigma_bins)+1))
    xs = torch.arange(K, device=device, dtype=torch.float32) - (K-1)/2
    k = torch.exp(-0.5 * (xs / float(sigma_bins))**2)
    k = k / (k.sum() + 1e-20)
    return k.view(1, 1, K)  # [out_channels=1, in_channels=1, K]

@torch.inference_mode()
def fit_densities_gpu_hist(
    bg_list: list[np.ndarray],
    fg_list: list[np.ndarray],
    num_points: int,
    sigma_bins: float,
    device: torch.device = torch.device("cuda"),
    batch_features: int = 64,
):
    """
    Fast GPU Stage-2:
      For each feature, make a histogram (B bins) of BG and FG samples, then
      smooth via conv1d with a Gaussian kernel (sigma in bins), and normalize to a PDF.

    Returns: x1_all, f1_all, x2_all, f2_all  (lists of numpy float32 with length = n_features)
    """
    assert len(bg_list) == len(fg_list)
    F_total = len(bg_list)
    B = int(num_points)
    # Prepare kernel once
    kernel = _gaussian_kernel_1d(sigma_bins=max(0.5, float(sigma_bins)), device=device)

    x1_all, f1_all, x2_all, f2_all = [], [], [], []

    # Process features in batches to reduce Python overhead without blowing VRAM
    for start in tqdm(range(0, F_total, batch_features), desc="GPU hist+density", total=(F_total + batch_features - 1)//batch_features):
        end = min(start + batch_features, F_total)
        for f in range(start, end):
            bg = bg_list[f]; fg = fg_list[f]
            # Handle empty classes gracefully
            if bg.size and fg.size:
                vmin = float(min(bg.min(), fg.min()))
                vmax = float(max(bg.max(), fg.max()))
            elif bg.size:
                vmin, vmax = float(bg.min()), float(bg.max())
            elif fg.size:
                vmin, vmax = float(fg.min()), float(fg.max())
            else:
                # degenerate: no samples
                x = np.linspace(0.0, 1.0, B, dtype=np.float32)
                zeros = np.zeros_like(x, dtype=np.float32)
                x1_all.append(x); f1_all.append(zeros)
                x2_all.append(x); f2_all.append(zeros)
                continue

            if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
                vmin, vmax = 0.0, 1.0

            # Build grid for this feature
            bin_width = (vmax - vmin) / B
            x_grid = torch.linspace(vmin, vmax, B, device=device, dtype=torch.float32)

            # Helper: bincount on GPU
            def _hist_1d(vals: np.ndarray) -> torch.Tensor:
                if vals.size == 0:
                    return torch.zeros(B, device=device, dtype=torch.float32)
                t = torch.from_numpy(vals).to(device=device, dtype=torch.float32)
                # map to indices [0..B-1]
                # clamp to avoid OOB when vals==vmax due to fp rounding
                idx = torch.clamp(((t - vmin) / (vmax - vmin + 1e-20) * (B - 1)).long(), 0, B - 1)
                h = torch.bincount(idx, minlength=B).to(torch.float32)
                return h

            h_bg = _hist_1d(bg)
            h_fg = _hist_1d(fg)

            # Smooth with Gaussian kernel (same σ for all features; good in practice)
            pad = (kernel.shape[-1] - 1) // 2
            s_bg = F.conv1d(h_bg.view(1,1,B), kernel, padding=pad).view(B)
            s_fg = F.conv1d(h_fg.view(1,1,B), kernel, padding=pad).view(B)

            # Convert to PDFs: counts / (N * bin_width)
            n_bg = max(1.0, float(bg.size))
            n_fg = max(1.0, float(fg.size))
            f_bg = s_bg / (n_bg * (bin_width + 1e-20))
            f_fg = s_fg / (n_fg * (bin_width + 1e-20))

            # Safety: nonnegative
            f_bg = torch.clamp(f_bg, min=0.0)
            f_fg = torch.clamp(f_fg, min=0.0)

            # Move to CPU numpy
            x_cpu = x_grid.detach().cpu().numpy().astype(np.float32)
            fbg_cpu = f_bg.detach().cpu().numpy().astype(np.float32)
            ffg_cpu = f_fg.detach().cpu().numpy().astype(np.float32)

            x1_all.append(x_cpu); f1_all.append(fbg_cpu)
            x2_all.append(x_cpu); f2_all.append(ffg_cpu)

            # free per-feature tensors
            del h_bg, h_fg, s_bg, s_fg, f_bg, f_fg, x_grid
        if device.type == "cuda":
            torch.cuda.empty_cache()

    return x1_all, f1_all, x2_all, f2_all


# -------------------- pack densities for GPU --------------------
@dataclass
class PackedDens:
    x1: Tensor; f1: Tensor; len1: Tensor
    x2: Tensor; f2: Tensor; len2: Tensor

def _pack(curves: List[np.ndarray], pad_val: float) -> Tuple[Tensor, Tensor]:
    lengths = np.array([len(c) for c in curves], dtype=np.int64)
    Lmax = int(lengths.max()) if lengths.size else 1
    F = len(curves)
    out = np.full((F, Lmax), pad_val, dtype=np.float32)
    for i, c in enumerate(curves):
        out[i, :len(c)] = c.astype(np.float32, copy=False)
    return torch.from_numpy(out), torch.from_numpy(lengths)

def pack_densities(x1_all, f1_all, x2_all, f2_all, device: torch.device) -> PackedDens:
    x1, len1 = _pack(x1_all, pad_val=np.inf)
    f1, _    = _pack(f1_all, pad_val=0.0)
    x2, len2 = _pack(x2_all, pad_val=np.inf)
    f2, _    = _pack(f2_all, pad_val=0.0)
    return PackedDens(x1=x1.to(device), f1=f1.to(device), len1=len1.to(device),
                      x2=x2.to(device), f2=f2.to(device), len2=len2.to(device))

@torch.no_grad()
def interp1d_per_feature(values: Tensor, xp: Tensor, fp: Tensor, lengths: Tensor) -> Tensor:
    B, N = values.shape
    Lmax = xp.shape[1]
    idx = torch.searchsorted(xp, values, right=False).clamp_(1, Lmax-1)
    left = idx - 1
    x0 = torch.gather(xp, 1, left); x1 = torch.gather(xp, 1, idx)
    y0 = torch.gather(fp, 1, left); y1 = torch.gather(fp, 1, idx)
    denom = (x1 - x0)
    t = torch.where(denom != 0, (values - x0) / denom, torch.zeros_like(values))
    out = y0 + (y1 - y0) * t
    xp_min = xp[:, :1]
    last_idx = (lengths - 1).clamp(min=0).view(B,1)
    xp_max = torch.gather(xp, 1, last_idx)
    in_range = (values >= xp_min) & (values <= xp_max)
    return torch.where(in_range, out, torch.zeros_like(out))


# -------------------- ROI classification (Stage 3) --------------------
class FeatureStacks:
    def __init__(self, filepaths: List[str]):
        self.handles = [TiffFile(fp) for fp in filepaths]
        self.paths = filepaths
    def read_slice_roi(self, z: int, ysl: slice, xsl: slice, idxs: List[int]) -> np.ndarray:
        tiles = []
        for i in idxs:
            tiles.append(self.handles[i].pages[z].asarray()[ysl, xsl])
        return np.stack(tiles, axis=0).astype(np.float32)
    def z_len(self, i: int) -> int:
        return len(self.handles[i].pages)
    def close(self):
        for h in self.handles:
            h.close()

@dataclass
class ROI:
    x_slice: slice
    y_slice: slice
    z_indices: List[int]

@dataclass
class Thresholds:
    bg_prob_gt: float = 0.4
    fg_prob_gt: float = 0.95

import torch.nn.functional as F
from concurrent.futures import ThreadPoolExecutor

def _detect_uniform_grid(dens, rtol=1e-5, atol=1e-6):
    """
    Check if x1/x2 are the same uniform grid for all features.
    Returns (is_uniform, vmin, inv_bw, L) in torch.float32 on the same device.
    """
    x = dens.x1  # [F,L]
    x0 = x[0]    # [L]
    if x0.numel() < 2:
        return (False, None, None, None)
    dx = x0[1:] - x0[:-1]
    # uniform spacing?
    if not torch.allclose(dx, dx[0].expand_as(dx), rtol=rtol, atol=atol):
        return (False, None, None, None)
    # same grid across all features?
    if not torch.allclose(x, x0.unsqueeze(0).expand_as(x), rtol=rtol, atol=atol):
        return (False, None, None, None)
    # x2 must match too
    if not torch.allclose(dens.x2, x0.unsqueeze(0).expand_as(dens.x2), rtol=rtol, atol=atol):
        return (False, None, None, None)
    vmin = x0[0].to(torch.float32)
    inv_bw = (1.0 / (x0[1] - x0[0])).to(torch.float32)
    L = x0.numel()
    return (True, vmin, inv_bw, L)

@torch.no_grad()
def _interp_uniform(values,  # [B,N] float
                    table,   # [B,L] float
                    vmin: torch.Tensor,  # scalar float32 on device
                    inv_bw: torch.Tensor,  # scalar float32 on device
                    L: int):
    """
    Linear interpolation on a shared, uniform grid:
      pos = (values - vmin)*inv_bw
      idx = floor(pos), t = pos - idx
      y = y[idx]*(1-t) + y[idx+1]*t, 0 outside grid
    values: [B,N] on device; table: [B,L] on device
    Returns: [B,N]
    """
    pos = (values - vmin) * inv_bw           # [B,N]
    idx_f = torch.floor(pos)
    idx = idx_f.to(torch.long)
    t = (pos - idx_f).clamp_(0, 1)           # [B,N]

    in_range = (idx >= 0) & (idx < (L - 1))
    idx = idx.clamp_(0, L - 2)

    # gather y0,y1
    y0 = torch.gather(table, 1, idx)
    y1 = torch.gather(table, 1, idx + 1)

    out = y0 + (y1 - y0) * t
    out = torch.where(in_range, out, torch.zeros_like(out))
    return out

@torch.no_grad()
def classify_roi(
    feature_paths: List[str],
    raw_tif: str,
    roi: ROI,
    dens: PackedDens,
    thr: Thresholds,
    out_dir: str,
    feature_batch: int,
    device: torch.device,
    dtype: torch.dtype = torch.float32,
    save_2D: bool = False,
):
    """
    - Computes all maps on GPU, but ACCUMULATES results on CPU (NumPy) as [Z, H, W].
    - Writes one 3D TIFF per output to out_dir.
    - If save_2D=True, also writes per-Z 2D TIFFs under out_dir/2D.
    """
    ensure_dir(out_dir)
    out_dir_2d = os.path.join(out_dir, "2D")
    if save_2D:
        ensure_dir(out_dir_2d)

    feats = FeatureStacks(feature_paths)
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

    H = roi.y_slice.stop - roi.y_slice.start
    W = roi.x_slice.stop - roi.x_slice.start
    Zout = len(roi.z_indices)
    F_total = len(feature_paths)
    batches = [list(range(i, min(i + feature_batch, F_total))) for i in range(0, F_total, feature_batch)]

    # ---- CPU stacks (accumulation on CPU, as requested) ----
    stack_export  = np.empty((Zout, H, W), dtype=np.float32)
    stack_exp2    = np.empty((Zout, H, W), dtype=np.float32)
    stack_exp3    = np.empty((Zout, H, W), dtype=np.float32)
    stack_exp4    = np.empty((Zout, H, W), dtype=np.float32)
    stack_exp5    = np.empty((Zout, H, W), dtype=np.float32)
    stack_rawcrop = np.empty((Zout, H, W), dtype=np.float32)

    with TiffFile(raw_tif) as tf_raw:
        for zi, z in enumerate(tqdm(roi.z_indices, desc="Processing Z")):
            # safety: ensure every feature has this z
            for i in range(F_total):
                if z >= len(feats.handles[i].pages):
                    raise RuntimeError(
                        f"Feature '{feature_paths[i]}' has only {len(feats.handles[i].pages)} slices; requested z={z}."
                    )

            # Per-Z accumulators on GPU
            sum_prob2 = torch.zeros((H, W), device=device, dtype=dtype)
            sum_prob1 = torch.zeros((H, W), device=device, dtype=dtype)
            sum_p1    = torch.zeros((H, W), device=device, dtype=dtype)
            sum_p2    = torch.zeros((H, W), device=device, dtype=dtype)
            cnt_bg    = torch.zeros((H, W), device=device, dtype=torch.int32)
            cnt_fg    = torch.zeros((H, W), device=device, dtype=torch.int32)

            for idxs in batches:
                B = len(idxs)
                vals_np = feats.read_slice_roi(z=z, ysl=roi.y_slice, xsl=roi.x_slice, idxs=idxs)  # [B,H,W] CPU
                vals = torch.from_numpy(vals_np).to(device=device, dtype=dtype)
                N = H * W
                vals_flat = vals.view(B, N)

                # Slice density tables for this batch
                x1b = dens.x1[idxs]; f1b = dens.f1[idxs]; l1b = dens.len1[idxs]
                x2b = dens.x2[idxs]; f2b = dens.f2[idxs]; l2b = dens.len2[idxs]

                p1 = interp1d_per_feature(vals_flat, x1b, f1b, l1b)  # [B,N]
                p2 = interp1d_per_feature(vals_flat, x2b, f2b, l2b)

                denom  = p1 + p2
                zero   = (denom == 0)
                prob1  = torch.where(zero, 0.5, p1 / denom).view(B, H, W)
                prob2  = torch.where(zero, 0.5, p2 / denom).view(B, H, W)
                p1     = p1.view(B, H, W)
                p2     = p2.view(B, H, W)

                sum_prob2 += prob2.sum(dim=0)
                sum_prob1 += prob1.sum(dim=0)
                sum_p1    += p1.sum(dim=0)
                sum_p2    += p2.sum(dim=0)
                cnt_bg    += (prob1 > thr.bg_prob_gt).sum(dim=0)
                cnt_fg    += (prob2 > thr.fg_prob_gt).sum(dim=0)

                # del vals, vals_np, vals_flat, x1b, f1b, l1b, x2b, f2b, l2b, p1, p2, prob1, prob2
                # if device.type == "cuda":
                #     torch.cuda.empty_cache()

            export_gpu = (cnt_fg > cnt_bg).to(dtype)

            # Move per-Z results to CPU NumPy and store into stacks
            stack_export[zi]  = export_gpu.detach().cpu().numpy()
            stack_exp2[zi]    = sum_prob2.detach().cpu().numpy()
            stack_exp3[zi]    = sum_p1.detach().cpu().numpy()
            stack_exp4[zi]    = sum_p2.detach().cpu().numpy()
            stack_exp5[zi]    = sum_prob1.detach().cpu().numpy()

            raw_tile = tf_raw.pages[z].asarray().astype(np.float32)
            raw_crop = raw_tile[roi.y_slice, roi.x_slice]
            stack_rawcrop[zi] = raw_crop  # already CPU

            # Optionally also write 2D files for this z
            if save_2D:
                z_matlab = z + 1
                imwrite(os.path.join(out_dir_2d, f"raw_crop_{z_matlab}.tif"), raw_crop, dtype=np.float32)
                imwrite(os.path.join(out_dir_2d, f"export_{z_matlab}.tif"),   stack_export[zi], dtype=np.float32)
                imwrite(os.path.join(out_dir_2d, f"export2_{z_matlab}.tif"),  stack_exp2[zi],   dtype=np.float32)
                imwrite(os.path.join(out_dir_2d, f"export3_{z_matlab}.tif"),  stack_exp3[zi],   dtype=np.float32)
                imwrite(os.path.join(out_dir_2d, f"export4_{z_matlab}.tif"),  stack_exp4[zi],   dtype=np.float32)
                imwrite(os.path.join(out_dir_2d, f"export5_{z_matlab}.tif"),  stack_exp5[zi],   dtype=np.float32)

            # cleanup GPU accumulators (now safely stored on CPU)
            # del export_gpu, sum_prob2, sum_prob1, sum_p1, sum_p2, cnt_bg, cnt_fg
            # if device.type == "cuda":
            #     torch.cuda.empty_cache()
            # gc.collect()

    feats.close()

    # ---- Write 3D stacks (Z,Y,X) to out_dir ----
    # Use BigTIFF to be safe for large volumes.
    imwrite(os.path.join(out_dir, "raw_crop.tif"), stack_rawcrop, dtype=np.float32, bigtiff=True)
    imwrite(os.path.join(out_dir, "export.tif"),   stack_export,  dtype=np.float32, bigtiff=True)
    imwrite(os.path.join(out_dir, "export2.tif"),  stack_exp2,    dtype=np.float32, bigtiff=True)
    imwrite(os.path.join(out_dir, "export3.tif"),  stack_exp3,    dtype=np.float32, bigtiff=True)
    imwrite(os.path.join(out_dir, "export4.tif"),  stack_exp4,    dtype=np.float32, bigtiff=True)
    imwrite(os.path.join(out_dir, "export5.tif"),  stack_exp5,    dtype=np.float32, bigtiff=True)



In [4]:
ap = argparse.ArgumentParser("SpatialDiNO pipeline with caching (skip Stage 1/2 when available)")
ap.add_argument("--features-root", required=True, help="Folder with feature .tif stacks (recursively).")
ap.add_argument("--seg-tif", required=True, help="Segmentation/mask .tif (same as MATLAB).")
ap.add_argument("--raw-tif", required=True, help="Raw volume .tif (same as MATLAB).")
ap.add_argument("--out-dir", required=True, help="Output directory.")
# Optional ranges (MATLAB inclusive). If omitted -> full extents.
ap.add_argument("--x-range", nargs=2, type=int, metavar=("START","END"))
ap.add_argument("--y-range", nargs=2, type=int, metavar=("START","END"))
ap.add_argument("--z-range", nargs=2, type=int, metavar=("START","END"))
# Perf knobs
ap.add_argument("--device", default=None, help="'cuda' or 'cpu' (auto if omitted).")
ap.add_argument("--feature-batch", type=int, default=32)
ap.add_argument("--kde-points", type=int, default=512)
ap.add_argument("--kde-max-samples", type=int, default=200000)
ap.add_argument("--kde-bandwidth", type=float, default=None)  # None=auto
# Caching
ap.add_argument("--samples-cache", default=None,
                help="Path to Stage1 samples cache (.npz). Default: <out_dir>/_stage1_samples.npz")
ap.add_argument("--densities-cache", default=None,
                help="Path to Stage2 densities cache (.npz). Default: <out_dir>/_stage2_densities.npz")
ap.add_argument("--rebuild-stage1", action="store_true", help="Force recompute Stage 1 (ignore samples cache).")
ap.add_argument("--rebuild-stage2", action="store_true", help="Force recompute Stage 2 (ignore densities cache).")
ap.add_argument("--seed", type=int, default=1337)
ap.add_argument("--density-method", default="kde",
                choices=["kde", "gpu-hist"],
                help="Stage-2 density estimator: 'kde' (CPU) or 'gpu-hist' (GPU histogram+Gaussian smoothing).")
ap.add_argument("--hist-sigma-bins", type=float, default=1.5,
                help="Stddev of Gaussian smoother in BINS for gpu-hist (typical 1.0–2.5).")
ap.add_argument(
    "--samples-compress",
    choices=["compressed", "none"],
    default="compressed",
    help="Compression for Stage 1 samples cache. 'none' is much faster but larger files."
)
ap.add_argument(
    "--densities-compress",
    choices=["compressed", "none"],
    default="compressed",
    help="Compression for Stage 2 densities cache. 'none' is faster but larger files."
)
ap.add_argument("--save_2D", "--save-2D", "--save-2d",
                dest="save_2D", action="store_true",
                help="Also save per-z 2D TIFFs under out_dir/2D (in addition to 3D stacks).")
ap.add_argument(
    "--valid-mask-tif",
    default=None,
    help="Optional TIFF mask: 1=valid voxel, 0=invalid. If provided, invalid voxels are excluded from Stage 1 sampling (and therefore Stage 2 densities).",
)

_StoreAction(option_strings=['--valid-mask-tif'], dest='valid_mask_tif', nargs=None, const=None, default=None, type=None, choices=None, required=False, help='Optional TIFF mask: 1=valid voxel, 0=invalid. If provided, invalid voxels are excluded from Stage 1 sampling (and therefore Stage 2 densities).', metavar=None)

In [5]:
# # seg_tif = "/nfs/scratch/Gustavo/ap2_using_ze_features/voronoi_from_cme_mask.tif"
# # cache_dir = "/nfs/scratch2/inacio/data/llsm/gu_seg/ap2/"

# seg_tif = "/nfs/scratch/Gustavo/nick_488_cme_mask/nick488_voronoi_from_cme_mask.tif"
# cache_dir = "/nfs/scratch2/inacio/data/llsm/gu_seg/nick488/"

# load_path = "/nfs/scratch2/inacio/data/llsm/spatialdino/nick488/"
# out_dir_parent = "/nfs/scratch2/inacio/data/llsm/gu_seg/nick488_movie/"

# args_list = []
# folders = sorted(os.listdir(load_path))
# # for t in range(1):
# for t in range(len(folders)):
#     features_root = os.path.join(load_path,folders[t],"features_tif")
#     raw_tif = os.path.join(load_path,folders[t],"volume_unnorm.tif")
#     out_dir = os.path.join(out_dir_parent, folders[t])
    
#     args = ap.parse_args([
#         "--features-root", features_root,
#         "--seg-tif", seg_tif,
#         "--raw-tif", raw_tif,
#         "--device", "cuda",
#         "--feature-batch", "64",
#         "--out-dir", out_dir,
#         "--density-method", "gpu-hist",
#         "--samples-compress", "none",
#         "--densities-compress", "none",
#         "--samples-cache", os.path.join(cache_dir, "_stage1_samples.npz"),
#         "--densities-cache", os.path.join(cache_dir, "_stage2_densities.npz"),
#     ])
#     args_list.append(args)

In [6]:
# seg_tif = "/nfs/scratch2/inacio/data/llsm/spatialdino/benchmark/2/masks/tumor.tif"
# valid_mask = "/nfs/scratch2/inacio/data/llsm/spatialdino/benchmark/2/masks/valid_voxels.tif"
# cache_dir = "/nfs/scratch2/inacio/data/llsm/gu_seg/benchmark/2/train_tumor/train001/"

# load_path = "/nfs/scratch2/inacio/data/llsm/spatialdino/benchmark/2/results_4x/test/"
# out_dir_parent = "/nfs/scratch2/inacio/data/llsm/gu_seg/benchmark/2/test_tumor/"

# args_list = []
# folders = sorted(os.listdir(load_path))
# for t in range(len(folders)):
#     features_root = os.path.join(load_path,folders[t],"features_tif")
#     raw_tif = os.path.join(load_path,folders[t],"volume_unnorm.tif")
#     out_dir = os.path.join(out_dir_parent, folders[t])
    
#     args = ap.parse_args([
#         "--features-root", features_root,
#         "--seg-tif", seg_tif,
#         "--raw-tif", raw_tif,
#         "--device", "cuda",
#         "--feature-batch", "64",
#         "--out-dir", out_dir,
#         "--density-method", "gpu-hist",
#         "--samples-compress", "none",
#         "--densities-compress", "none",
#         "--samples-cache", os.path.join(cache_dir, "_stage1_samples.npz"),
#         "--densities-cache", os.path.join(cache_dir, "_stage2_densities.npz"),
#         "--valid-mask-tif", valid_mask,
#     ])
#     args_list.append(args)

In [11]:
# seg_tif = "/nfs/scratch/Gustavo/ap2_using_ze_features/voronoi_from_cme_mask.tif"
# cache_dir = "/nfs/scratch2/inacio/data/llsm/gu_seg/ap2/"
seg_tif = "/nfs/scratch2/inacio/data/llsm/spatialdino/benchmark/5_alt/raw/mask.tif"
cache_dir = "/nfs/scratch2/inacio/data/llsm/gu_seg/benchmark/5_alt/train_with_right/"

load_path = "/nfs/scratch2/inacio/data/llsm/spatialdino/benchmark/5/results/test/"
out_dir_parent = "/nfs/scratch2/inacio/data/llsm/gu_seg/benchmark/5_alt/test/"

args_list = []
folders = sorted(os.listdir(load_path))
for t in range(len(folders)):
    features_root = os.path.join(load_path,folders[t],"features_tif")
    raw_tif = os.path.join(load_path,folders[t],"volume_unnorm.tif")
    out_dir = os.path.join(out_dir_parent, folders[t])
    
    args = ap.parse_args([
        "--features-root", features_root,
        "--seg-tif", seg_tif,
        "--raw-tif", raw_tif,
        "--device", "cuda",
        "--feature-batch", "64",
        "--out-dir", out_dir,
        "--density-method", "gpu-hist",
        "--samples-compress", "none",
        "--densities-compress", "none",
        "--samples-cache", os.path.join(cache_dir, "_stage1_samples.npz"),
        "--densities-cache", os.path.join(cache_dir, "_stage2_densities.npz"),
        # "--valid-mask-tif", valid_mask,
    ])
    args_list.append(args)

In [12]:
start = time()

def main(args_list):

    args = args_list[0]

    rng = np.random.default_rng(args.seed)
    ensure_dir(args.out_dir)
    
    # Resolve default cache paths
    samples_cache = args.samples_cache or os.path.join(args.out_dir, "_stage1_samples.npz")
    densities_cache = args.densities_cache or os.path.join(args.out_dir, "_stage2_densities.npz")
    
    # Discover features and scan shapes
    feature_paths = find_tiffs_sorted_by_last_number(args.features_root)
    shape = scan_feature_shapes(feature_paths)

    # ---------------- Stage 1: load or compute samples ----------------
    if (not args.rebuild_stage1) and os.path.exists(samples_cache):
        print(f"[cache] Loading Stage 1 samples from {samples_cache}")
        bg_list, fg_list, saved_paths, saved_names = load_samples_npz(samples_cache)
        # Reorder to current feature order if necessary
        try:
            bg_list, fg_list = reorder_lists_to_current(
                feature_paths, saved_paths, saved_names, [bg_list, fg_list]
            )
        except RuntimeError as e:
            raise RuntimeError(f"Samples cache mismatch: {e}")
    else:
        print("[compute] Stage 1: collecting samples (this reads features + seg)")
        seg = read_tiff_volume(args.seg_tif).astype(np.int64)
        bg_label = int(np.min(seg))
        Zc = seg.shape[0]
        seg_c = seg[:Zc]
        if args.valid_mask_tif:
            valid_c = load_valid_mask(args.valid_mask_tif, Z=Zc, Y=shape.y, X=shape.x).astype(bool, copy=False)
        else:
            valid_c = np.ones((Zc, shape.y, shape.x), dtype=bool)
        bg_mask = (seg_c == bg_label) & valid_c
        fg_mask = (seg_c != bg_label) & valid_c
        bg_list, fg_list = collect_samples_all_features(
            feature_paths, bg_mask, fg_mask, max_samples_per_class=args.kde_max_samples, rng=rng
        )
        print(f"[save] Writing Stage 1 samples to {samples_cache}")
        save_samples_npz(
            samples_cache,
            bg_list, fg_list, feature_paths,
            compress=(args.samples_compress == "compressed"),
        )
    
    # ---------------- Stage 2: load or compute densities ----------------
    if (not args.rebuild_stage2) and os.path.exists(densities_cache):
        print(f"[cache] Loading Stage 2 densities from {densities_cache}")
        x1_all, f1_all, x2_all, f2_all, saved_paths_d, saved_names_d = load_densities_npz(densities_cache)
        # Reorder to current feature order if necessary
        try:
            x1_all, f1_all, x2_all, f2_all = reorder_lists_to_current(
                feature_paths, saved_paths_d, saved_names_d, [x1_all, f1_all, x2_all, f2_all]
            )
        except RuntimeError as e:
            raise RuntimeError(f"Densities cache mismatch: {e}")
    else:
        print(f"[compute] Stage 2: fitting densities ({args.density_method}) from Stage 1 samples")
        if args.density_method == "kde":
            x1_all, f1_all, x2_all, f2_all = fit_densities_from_samples(
                bg_list, fg_list, num_points=args.kde_points, bandwidth=args.kde_bandwidth
            )
        elif args.density_method == "gpu-hist":
            device = torch.device(args.device if args.device is not None
                                  else ("cuda" if torch.cuda.is_available() else "cpu"))
            if device.type != "cuda":
                print("[warn] gpu-hist requested but CUDA not available; running on CPU (still faster than KDE).")
            x1_all, f1_all, x2_all, f2_all = fit_densities_gpu_hist(
                bg_list, fg_list,
                num_points=args.kde_points,
                sigma_bins=args.hist_sigma_bins,
                device=device,
                batch_features=64  # adjust for your VRAM
            )
        else:
            raise ValueError(f"Unknown density method: {args.density_method}")
        print(f"[save] Writing Stage 2 densities to {densities_cache}")
        save_densities_npz(
            densities_cache,
            x1_all, f1_all, x2_all, f2_all, feature_paths,
            compress=(args.densities_compress == "compressed"),
        )
    
    # ---------------- Stage 3: classify ROI on GPU ----------------
    device = torch.device(args.device if args.device is not None else ("cuda" if torch.cuda.is_available() else "cpu"))
    dens = pack_densities(x1_all, f1_all, x2_all, f2_all, device=device)
    

    for args in args_list:
        feature_paths = find_tiffs_sorted_by_last_number(args.features_root)
        shape = scan_feature_shapes(feature_paths)
        
        with TiffFile(args.raw_tif) as tf_raw:
            Z_raw = len(tf_raw.pages)
            y_raw, x_raw = tf_raw.pages[0].asarray().shape
        if (y_raw, x_raw) != (shape.y, shape.x):
            raise ValueError(f"Raw Y/X {y_raw,x_raw} != feature Y/X {shape.y,shape.x}")
        
        # ROI defaults to full extents if omitted; clamp to bounds
        def clamp_or_default(range_flag, size, label):
            if range_flag is None:
                return (1, size)
            a, b = range_flag
            return clamp_inclusive_range(a, b, 1, size)
        
        x1,x2 = clamp_or_default(args.x_range, shape.x, "X")
        y1,y2 = clamp_or_default(args.y_range, shape.y, "Y")
        z1,z2 = clamp_or_default(args.z_range, Z_raw, "Z")
        xsl = matlab_range_to_py_slice(x1, x2)
        ysl = matlab_range_to_py_slice(y1, y2)
        z_indices = list(range(z1-1, z2))  # 0-based inclusive

        roi = ROI(x_slice=xsl, y_slice=ysl, z_indices=z_indices)
    
        
        classify_roi(
            feature_paths=feature_paths,
            raw_tif=args.raw_tif,
            roi=roi,
            dens=dens,
            thr=Thresholds(bg_prob_gt=0.4, fg_prob_gt=0.95),
            out_dir=args.out_dir,
            feature_batch=int(args.feature_batch),
            device=device,
            dtype=torch.float32,
            save_2D=bool(args.save_2D),
        )
    print("✅ Done.")

main(args_list)

print(time()-start)

[cache] Loading Stage 1 samples from /nfs/scratch2/inacio/data/llsm/gu_seg/benchmark/5_alt/train_with_right/_stage1_samples.npz
[cache] Loading Stage 2 densities from /nfs/scratch2/inacio/data/llsm/gu_seg/benchmark/5_alt/train_with_right/_stage2_densities.npz


Processing Z: 100%|███████████████████████████████████████████████████████████████████| 208/208 [04:22<00:00,  1.26s/it]


✅ Done.
274.795129776001
